In [3]:
# Step 4 Demo: Compare multiple classification models
# Goal: predict the recommended vertical wind turbine product
# from the real survey dataset using the same preprocessing
# and same train/test split for all models.

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Optional: install first if needed
# pip install xgboost
from xgboost import XGBClassifier

# ----------------------------------------
# 1. Load the actual survey file
# ----------------------------------------
df = pd.read_excel("Survey Results _02.xls")

# Keep only valid response rows
df = df[df["Timestamp"].notna()].copy()

# ----------------------------------------
# 2. Convert Neutrosophic survey responses
# ----------------------------------------
response_cols = df.columns[7:]

response_map = {
    "Truth (T)": 0.70,
    "Indeterminacy (I)": 0.20,
    "Falsity (F)": 0.10
}

for col in response_cols:
    df[col] = df[col].map(response_map)

# ----------------------------------------
# 3. Aggregate the 7 design criteria
# ----------------------------------------
criteria = {
    "Social": list(response_cols[0:5]),
    "Economic": list(response_cols[5:10]),
    "Environmental": list(response_cols[10:15]),
    "Technical": list(response_cols[15:20]),
    "Installation": list(response_cols[20:25]),
    "Policy": list(response_cols[25:30]),
    "UserExperience": list(response_cols[30:35]),
}

for crit, cols in criteria.items():
    df[crit] = df[cols].mean(axis=1)

# ----------------------------------------
# 4. Create a demo recommendation target
# ----------------------------------------
# Since the survey file has no direct "recommended product" column,
# we create one using a product-fit score based on survey priorities.

type_boost = {
    "Household": 0.00,
    "Others": 0.03,
    "Small business owner": 0.08,
    "Non-government organization (NGO)": 0.12,
    "Household;Small business owner": 0.07
}

df["power_fit_score"] = (
    0.35 * df["Technical"] +
    0.20 * df["Installation"] +
    0.20 * df["UserExperience"] +
    0.15 * df["Economic"] +
    0.10 * df["Social"] +
    df["Type of Respondent"].map(type_boost).fillna(0.02)
)

product_labels = [
    "Lantern Wind Turbine - 10-50W",
    "Candle Light - 50W",
    "Standard Wind Turbine - 600W",
    "Tri-blade Wind Turbine - 200W-2KW",
    "Stackable Wind Turbine - 5KW+"
]

# Convert score into 5 product classes
df["Recommended_Product"] = pd.qcut(
    df["power_fit_score"],
    q=5,
    labels=product_labels
).astype(str)

# ----------------------------------------
# 5. Select input features
# ----------------------------------------
feature_cols = [
    "Province",
    "Age Bracket (Years Old)",
    "Type of Respondent",
    "\nCurrent energy source ",
    "Social",
    "Economic",
    "Environmental",
    "Technical",
    "Installation",
    "Policy",
    "UserExperience",
    "power_fit_score"
]

X = df[feature_cols]
y = df["Recommended_Product"]

# Encode target labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# ----------------------------------------
# 6. Train/test split
# ----------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

# ----------------------------------------
# 7. Preprocessing pipeline
# ----------------------------------------
categorical_cols = [
    "Province",
    "Age Bracket (Years Old)",
    "Type of Respondent",
    "\nCurrent energy source "
]

numeric_cols = [
    "Social",
    "Economic",
    "Environmental",
    "Technical",
    "Installation",
    "Policy",
    "UserExperience",
    "power_fit_score"
]

preprocessor = ColumnTransformer([
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_cols),

    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numeric_cols)
])

# ----------------------------------------
# 8. Define models
# ----------------------------------------
models = {
    "Logistic Regression": LogisticRegression(max_iter=3000),
    "Decision Tree": DecisionTreeClassifier(max_depth=4, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42),
    "XGBoost": XGBClassifier(
        n_estimators=50,
        max_depth=3,
        learning_rate=0.1,
        objective="multi:softprob",
        eval_metric="mlogloss",
        random_state=42
    )
}

# ----------------------------------------
# 9. Train and compare models
# ----------------------------------------
results = []

for model_name, model in models.items():
    pipeline = Pipeline([
        ("preprocess", preprocessor),
        ("classifier", model)
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="macro")

    results.append({
        "Model": model_name,
        "Accuracy": acc,
        "F1_macro": f1
    })

results_df = pd.DataFrame(results).sort_values(by="Accuracy", ascending=False)

print(results_df)

# Optional: decode predicted classes for inspection
# print(label_encoder.inverse_transform(y_pred))

                 Model  Accuracy  F1_macro
1        Decision Tree  1.000000  1.000000
3              XGBoost  1.000000  1.000000
0  Logistic Regression  0.571429  0.466667
2        Random Forest  0.571429  0.393333
